# Chapitre 5 - Exercices 5.1 à 5.5

Base des puissances tronquées, propriétés des B-splines, variance ponctuelle (figure 5.3),
contraintes des splines naturelles et filtrage des phonèmes pour une analyse
discriminante quadratique.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import (truncated_power_basis, bspline_basis, augmented_knots,
                              natural_spline_basis)
set_style()

## Exercice 5.1 : La base des puissances tronquées

**Énoncé.** Montrer que les fonctions de l'éq. (5.3),
$h_1 = 1$, $h_2 = X$, $h_3 = X^2$, $h_4 = X^3$, $h_5 = (X - \xi_1)_+^3$,
$h_6 = (X - \xi_2)_+^3$, forment une base de l'espace des splines cubiques de nœuds
$\xi_1 < \xi_2$.

**Rappel.** Une spline cubique de nœuds $\xi_1 < \xi_2$ est une fonction $f : \mathbb R
\to \mathbb R$ qui est un polynôme de degré $\le 3$ sur chacun des intervalles
$(-\infty, \xi_1]$, $[\xi_1, \xi_2]$, $[\xi_2, +\infty)$, et de classe $C^2$ (continue,
ainsi que $f'$ et $f''$). L'ensemble $\mathcal S$ de ces fonctions est un espace vectoriel
(stable par combinaison linéaire). Il faut montrer trois choses : les $h_m$ sont dans
$\mathcal S$, elles sont libres, et elles engendrent $\mathcal S$.

**1. Les $h_m$ sont des splines cubiques.** C'est clair pour $1, X, X^2, X^3$. Pour
$g(X) = (X - \xi)_+^3$ : $g$ vaut $0$ sur $(-\infty, \xi]$ et $(X - \xi)^3$ sur
$[\xi, +\infty)$, deux polynômes de degré $\le 3$. Ses dérivées à gauche de $\xi$ sont
nulles, à droite elles valent $3(X-\xi)^2$, $6(X - \xi)$, qui tendent vers $0$ en $\xi$.
Donc $g, g', g''$ sont continues en $\xi$ : $g \in C^2$. (La dérivée troisième saute de
$0$ à $6$.)

**2. Liberté.** Supposons $\sum_{j=0}^{3}\beta_jX^j + \theta_1(X-\xi_1)_+^3 +
\theta_2(X-\xi_2)_+^3 = 0$ pour tout $X$.

- Sur $(-\infty, \xi_1)$, les deux puissances tronquées sont nulles : le polynôme
  $\sum\beta_jX^j$ s'annule sur un intervalle, donc tous ses coefficients sont nuls.
- Sur $(\xi_1, \xi_2)$, il reste $\theta_1(X - \xi_1)^3 = 0$, donc $\theta_1 = 0$.
- Sur $(\xi_2, +\infty)$, il reste $\theta_2(X - \xi_2)^3 = 0$, donc $\theta_2 = 0$.

**3. Les $h_m$ engendrent $\mathcal S$.** Soit $f \in \mathcal S$ et $p_0$ le polynôme
égal à $f$ sur $(-\infty, \xi_1]$ ; posons $g = f - p_0 \in \mathcal S$, nulle sur
$(-\infty, \xi_1]$.

- Sur $[\xi_1, \xi_2]$, $g$ est un polynôme $q$ de degré $\le 3$ et, par continuité de
  $g, g', g''$ en $\xi_1$, $q(\xi_1) = q'(\xi_1) = q''(\xi_1) = 0$. La formule de Taylor
  (exacte pour un polynôme de degré 3) donne $q(X) = \frac{q'''(\xi_1)}{6}(X-\xi_1)^3 =
  c_1(X - \xi_1)^3$.
- La fonction $g_1 = g - c_1(X - \xi_1)_+^3$ est dans $\mathcal S$ et nulle sur
  $(-\infty, \xi_2]$. Le même argument en $\xi_2$ donne $g_1 = c_2(X - \xi_2)_+^3$.

Ainsi $f = p_0 + c_1(X - \xi_1)_+^3 + c_2(X - \xi_2)_+^3$ : $\dim\mathcal S = 6$. Le même
raisonnement montre qu'une spline d'ordre $M$ à $K$ nœuds s'écrit dans la base
$\{X^{j}\}_{j<M} \cup \{(X - \xi_k)_+^{M-1}\}_{k \le K}$, de dimension $M + K$.

**Vérification numérique.** Construisons une spline cubique « au hasard » par une autre
voie (combinaison aléatoire de B-splines, cf. cours 1) et vérifions qu'elle s'écrit
**exactement** dans la base (5.3).

In [2]:
rng = np.random.RandomState(0)
xi = np.array([0.3, 0.7])
x = np.linspace(0, 1, 500)
f_rand = bspline_basis(x, xi, 0, 1) @ rng.randn(6)
H = truncated_power_basis(x, xi)
coef, *_ = np.linalg.lstsq(H, f_rand, rcond=None)
print("rang de H :", np.linalg.matrix_rank(H), "| résidu max :", np.abs(H @ coef - f_rand).max())

rang de H : 6 | résidu max : 2.0372592501871623e-14


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for j in range(6):
    ax.plot(x, coef[j] * H[:, j], lw=1.2, color=COLORS[j], label="$\\hat c_{%d} h_{%d}$" % (j + 1, j + 1))
ax.plot(x, f_rand, color="k", lw=2.5, ls="--", label="spline $f$")
for k in xi:
    ax.axvline(k, color=GRAY, lw=0.8, ls=":")
ax.set_xlabel("$X$")
ax.set_ylabel("contribution")
ax.set_title("Décomposition d'une spline cubique dans la base (5.3)")
ax.legend(ncol=2, fontsize=8)
plt.show()

> **À retenir.** Les puissances tronquées encodent directement le saut de la dérivée
> d'ordre $M - 1$ en chaque nœud ; la preuve « on retire le polynôme de gauche, puis on
> avance de nœud en nœud » montre que la dimension vaut $M + K$.

## Exercice 5.2 : Propriétés des B-splines

**Énoncé.** Soit $B_{i,M}$ la B-spline d'ordre $M$ définie par la récurrence
(5.77)-(5.78) (rappelée ci-dessous) sur la suite de nœuds augmentée $\tau$. Montrer par
récurrence :

(a) $B_{i,M}(x) = 0$ pour $x \notin [\tau_i, \tau_{i+M}]$ (support d'au plus $M+1$ nœuds) ;
(b) $B_{i,M}(x) > 0$ pour $x \in (\tau_i, \tau_{i+M})$ ;
(c) $\sum_{i=1}^{K+M}B_{i,M}(x) = 1$ pour $x \in [\xi_0, \xi_{K+1}]$ ;
(d) $B_{i,M}$ est polynomiale par morceaux de degré $M - 1$ sur $[\xi_0, \xi_{K+1}]$,
    avec des cassures seulement aux nœuds ;
(e) une B-spline d'ordre $M$ est la densité d'une somme (convolution) de $M$ variables
    uniformes.

**Rappels.** $B_{i,1}(x) = \mathbf 1(\tau_i \le x < \tau_{i+1})$, et pour $m \ge 2$

$$
B_{i,m}(x) = \underbrace{\frac{x - \tau_i}{\tau_{i+m-1} - \tau_i}}_{w_{i,m}(x)}B_{i,m-1}(x)
 + \underbrace{\frac{\tau_{i+m} - x}{\tau_{i+m} - \tau_{i+1}}}_{\bar w_{i,m}(x)}B_{i+1,m-1}(x),
$$

avec la convention qu'un terme dont le dénominateur est nul est nul (dans ce cas, la
B-spline qu'il multiplie est identiquement nulle, comme on le verra en (a)).

**(a) Support.** Pour $m = 1$, $B_{i,1}$ est nulle hors de $[\tau_i, \tau_{i+1})$.
Supposons la propriété vraie à l'ordre $m - 1$ : $B_{i,m-1}$ est nulle hors de
$[\tau_i, \tau_{i+m-1}]$ et $B_{i+1,m-1}$ hors de $[\tau_{i+1}, \tau_{i+m}]$. Ces deux
intervalles sont inclus dans $[\tau_i, \tau_{i+m}]$, donc hors de cet intervalle les deux
termes de la récurrence sont nuls : $B_{i,m}(x) = 0$. Pour $M = 4$, le support
$[\tau_i, \tau_{i+4}]$ fait intervenir 5 nœuds.

En particulier, si $\tau_i = \tau_{i+m}$, $B_{i,m} \equiv 0$ (sauf sur un point), ce qui
justifie la convention $0/0 = 0$.

**(b) Positivité.** Montrons d'abord $B_{i,m} \ge 0$ partout. Vrai pour $m = 1$. Si c'est
vrai à l'ordre $m - 1$ : là où $B_{i,m-1}(x) \ne 0$ on a $x \ge \tau_i$ par (a), donc
$w_{i,m}(x) \ge 0$ ; là où $B_{i+1,m-1}(x) \ne 0$ on a $x \le \tau_{i+m}$, donc
$\bar w_{i,m}(x) \ge 0$. Les deux termes sont $\ge 0$.

Stricte positivité, pour des nœuds distincts (cas simple) : par récurrence, $B_{i,m-1} > 0$
sur $(\tau_i, \tau_{i+m-1})$ et $B_{i+1,m-1} > 0$ sur $(\tau_{i+1}, \tau_{i+m})$. Soit
$x \in (\tau_i, \tau_{i+m})$ :

- si $x < \tau_{i+m-1}$ : $w_{i,m}(x) > 0$ (car $x > \tau_i$) et $B_{i,m-1}(x) > 0$, le
  premier terme est $> 0$ ;
- si $x > \tau_{i+1}$ : de même le second terme est $> 0$.

Pour $m \ge 3$, $\tau_{i+1} < \tau_{i+m-1}$ et l'un des deux cas a toujours lieu. Pour
$m = 2$, le seul point non couvert est $x = \tau_{i+1}$, où un calcul direct donne
$B_{i,2}(\tau_{i+1}) = \bar w_{i,2}(\tau_{i+1})B_{i+1,1}(\tau_{i+1}) = 1 > 0$. Avec des
nœuds répétés, le même argument s'applique sur les intervalles non dégénérés.

**(c) Partition de l'unité.** Notons $n_m = K + 2M - m$ le nombre de B-splines d'ordre $m$.
En sommant la récurrence et en posant $j = i + 1$ dans le second terme :

$$
\sum_{i=1}^{n_m}B_{i,m}(x) = \sum_{i=1}^{n_m}w_{i,m}(x)B_{i,m-1}(x)
  + \sum_{j=2}^{n_m+1}\bar w_{j-1,m}(x)B_{j,m-1}(x).
$$

Or $w_{j,m}(x) + \bar w_{j-1,m}(x) = \frac{x - \tau_j}{\tau_{j+m-1} - \tau_j} +
\frac{\tau_{j+m-1} - x}{\tau_{j+m-1} - \tau_j} = 1$ (les deux fractions ont le même
dénominateur). Les deux termes « orphelins » sont $B_{1,m-1}$ (support
$[\tau_1, \tau_m] = \{\xi_0\}$ quand $m \le M$) et $B_{n_{m-1},m-1}$ (support réduit à
$\{\xi_{K+1}\}$) : ils sont nuls sur l'intervalle. Donc
$\sum_iB_{i,m} = \sum_jB_{j,m-1}$ sur $[\xi_0, \xi_{K+1}]$, et par récurrence
$= \sum_iB_{i,1} = 1$, car chaque $x$ appartient à exactement un intervalle
$[\tau_i, \tau_{i+1})$ (le point $\xi_{K+1}$ étant rattaché au dernier intervalle).

**(d) Polynômes par morceaux.** $B_{i,1}$ est constante sur chaque intervalle entre
nœuds consécutifs. Si $B_{i,m-1}$ et $B_{i+1,m-1}$ sont polynomiales de degré $\le m-2$
sur chacun de ces intervalles, alors $B_{i,m}$, combinaison à coefficients **affines**
$w, \bar w$, est polynomiale de degré $\le m - 1$ sur ces mêmes intervalles. Les seuls
points de cassure possibles sont donc les nœuds. (On montre de plus que $B_{i,M}$ est
de classe $C^{M-2}$ en un nœud simple : c'est une spline.)

**(e) Convolution d'uniformes.** Prenons des nœuds **équidistants** de pas 1,
$\tau_i = i$. Par invariance par translation, $B_{i,m}(x) = N_m(x - i)$ où
$N_m = B_{0,m}$ a pour support $[0, m]$, et la récurrence devient

$$
N_m(x) = \frac{x}{m-1}N_{m-1}(x) + \frac{m - x}{m-1}N_{m-1}(x - 1). \tag{R}
$$

**Lemme.** Pour $m \ge 2$ et $x$ hors des nœuds : $N_m'(x) = N_{m-1}(x) - N_{m-1}(x-1)$.